# CIFAR-10 Image Classification Model
## Performance Analysis across Architectures & Training Strategies

In this notebook, we build and analyze image classification models on the CIFAR-10 dataset. We will compare:
1. A Baseline Artificial Neural Network (ANN)
2. A Spatial Convolutional Neural Network (CNN)
3. A Data-Augmented CNN

We will also address all the requested student tasks:
- **Task 1:** Increase dense layout configurations.
- **Task 2:** Scale up filter sizes (32->64->128).
- **Task 3:** Increase training to 20 epochs.
- **Task 4:** Integrate EarlyStopping.
- **Task 5:** Execute the augmented network training run.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Print TensorFlow version
print(f"TensorFlow Version: {tf.__version__}")

## 1. Data Loading & Preprocessing
Load the standard CIFAR-10 database and normalize the pixel integers (0-255 -> 0.0-1.0) to stabilize gradient updates.

In [ ]:
# Load CIFAR-10 dataset
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Normalize pixel values
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

# Define class names for reference
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

## 2. Early Stopping Callback
**Task 4**: Integrate EarlyStopping to prevent overfitting during the 20 epochs of training.

In [ ]:
# Define EarlyStopping callback
early_stopping = callbacks.EarlyStopping(
    monitor='val_accuracy', 
    patience=5, 
    restore_best_weights=True,
    verbose=1
)

## 3. Baseline ANN Model
Build a baseline ANN model on flat vectors using Dense and Dropout layers.
**Task 1**: Increase dense layout configurations. We use a deeper architecture with 512 and 256 units.

In [ ]:
# Build ANN Model
ann_model = models.Sequential([
    layers.Flatten(input_shape=(32, 32, 3)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

ann_model.summary()

**Task 3**: Increase training to 20 epochs.

In [ ]:
print("Training ANN...")
ann_history = ann_model.fit(
    X_train, y_train, 
    epochs=20, 
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    batch_size=64
)

## 4. Spatial CNN Architecture
Build a spatial CNN architecture using Conv2D, BatchNormalization, MaxPooling2D, Flatten, and Dense layers.
**Task 2**: Scale up filter sizes (32 -> 64 -> 128).

In [ ]:
# Build CNN Model
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cnn_model.summary()

In [ ]:
print("Training Baseline CNN...")
cnn_history = cnn_model.fit(
    X_train, y_train, 
    epochs=20, 
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    batch_size=64
)

## 5. Data-Augmented CNN
Build an advanced data-augmented network variant.
**Task 5**: Execute the augmented network training run using RandomFlip, RandomRotation, and RandomZoom.

In [ ]:
# Define data augmentation layers
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal", input_shape=(32, 32, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Build Augmented CNN Model
aug_cnn_model = models.Sequential([
    data_augmentation,
    
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

aug_cnn_model.summary()

In [ ]:
print("Training Augmented CNN...")
aug_cnn_history = aug_cnn_model.fit(
    X_train, y_train, 
    epochs=20, 
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    batch_size=64
)

## 6. Validation Curve Chart
Plot validation accuracy tracking curves for all pipelines on a single chart.

In [ ]:
plt.figure(figsize=(12, 8))

# ANN curves
plt.plot(ann_history.history['val_accuracy'], label='ANN Validation Accuracy', linestyle='--', color='blue')

# CNN curves
plt.plot(cnn_history.history['val_accuracy'], label='CNN Validation Accuracy', linestyle='-', color='orange')

# Augmented CNN curves
plt.plot(aug_cnn_history.history['val_accuracy'], label='Augmented CNN Validation Accuracy', linestyle='-', color='green', linewidth=2)

plt.title('Validation Accuracy Comparison across Models (CIFAR-10)')
plt.xlabel('Epochs')
plt.ylabel('Validation Accuracy')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Final Output Comparison
Create a final output comparison dataframe contrasting test accuracy scores between the model variants.

In [ ]:
# Evaluate models on test set
print("Evaluating ANN...")
_, ann_test_acc = ann_model.evaluate(X_test, y_test, verbose=0)

print("Evaluating Baseline CNN...")
_, cnn_test_acc = cnn_model.evaluate(X_test, y_test, verbose=0)

print("Evaluating Augmented CNN...")
_, aug_cnn_test_acc = aug_cnn_model.evaluate(X_test, y_test, verbose=0)

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Model Architecture': ['Baseline ANN', 'Spatial CNN', 'Augmented Spatial CNN'],
    'Test Accuracy': [ann_test_acc, cnn_test_acc, aug_cnn_test_acc],
    'Max Training Epochs': [20, 20, 20],
    'Early Stopping': ['Yes', 'Yes', 'Yes']
})

# Format percentage
comparison_df['Test Accuracy'] = (comparison_df['Test Accuracy'] * 100).map('{:.2f}%'.format)

# Display DataFrame
print("\n--- Final Model Performance Comparison ---")
display(comparison_df)